In [3]:
%pip install polars pyarrow

In [9]:
import polars as pl

train_tx = pl.read_parquet("data/train_transactions.parquet")
test_tx = pl.read_parquet("data/test_transactions.parquet")

print(train_tx.shape)
print(test_tx.shape)

(6987663, 5)
(3027575, 5)


In [34]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [5]:
"""
Precompute every summary CSV the Streamlit EDA app reads from eda_data/.
Run this once (locally) against the real competition data before deploying
the app, and commit the resulting eda_data/*.csv files alongside app.py.

Only small, aggregated summaries are written -- never raw transaction rows --
so the deployed app stays lightweight and never needs the multi-million-row
parquet files at runtime.
"""

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
OUT_DIR = Path("eda_data")
OUT_DIR.mkdir(exist_ok=True)

train_signals = pd.read_csv(DATA_DIR / "train_signals.csv", parse_dates=["signal_sanasi"])
test_signals = pd.read_csv(DATA_DIR / "test_signals.csv", parse_dates=["signal_sanasi"])
train_tx = pd.read_parquet(DATA_DIR / "train_transactions.parquet")
test_tx = pd.read_parquet(DATA_DIR / "test_transactions.parquet")

train_tx["tranzaksiya_vaqti"] = pd.to_datetime(train_tx["tranzaksiya_vaqti"])
test_tx["tranzaksiya_vaqti"] = pd.to_datetime(test_tx["tranzaksiya_vaqti"])

all_tx = pd.concat([train_tx, test_tx], ignore_index=True)
all_signals = pd.concat(
    [train_signals[["signal_id", "signal_sanasi"]], test_signals[["signal_id", "signal_sanasi"]]],
    ignore_index=True,
)

# ---------------------------------------------------------------- #
# 1. target_distribution.csv
# ---------------------------------------------------------------- #
target_distribution = (
    train_signals["eskalatsiya"].value_counts().rename_axis("eskalatsiya")
    .reset_index(name="count")
)
target_distribution.to_csv(OUT_DIR / "target_distribution.csv", index=False)

# ---------------------------------------------------------------- #
# 2. transaction_direction.csv
# ---------------------------------------------------------------- #
transaction_direction = (
    all_tx["kirim_chiqim"].value_counts().rename_axis("kirim_chiqim")
    .reset_index(name="count")
)
transaction_direction.to_csv(OUT_DIR / "transaction_direction.csv", index=False)

# ---------------------------------------------------------------- #
# 3. transaction_types.csv
# ---------------------------------------------------------------- #
transaction_types = (
    all_tx["tranzaksiya_turi"].value_counts().rename_axis("tranzaksiya_turi")
    .reset_index(name="count")
)
transaction_types.to_csv(OUT_DIR / "transaction_types.csv", index=False)

# ---------------------------------------------------------------- #
# 4. activity_over_time.csv  (REQUIRED example: "transaction activity over time")
# ---------------------------------------------------------------- #
activity_over_time = (
    all_tx.assign(period=all_tx["tranzaksiya_vaqti"].dt.to_period("M").astype(str))
    .groupby("period").size().reset_index(name="count")
    .sort_values("period")
)
activity_over_time.to_csv(OUT_DIR / "activity_over_time.csv", index=False)

# ---------------------------------------------------------------- #
# 5. pre_signal_activity.csv  (REQUIRED example: "activity immediately before a signal")
# ---------------------------------------------------------------- #
tx_with_signal = all_tx.merge(all_signals, on="signal_id", how="inner")
tx_with_signal = tx_with_signal[tx_with_signal["tranzaksiya_vaqti"] < tx_with_signal["signal_sanasi"]]
days_before = (
    (tx_with_signal["signal_sanasi"] - tx_with_signal["tranzaksiya_vaqti"]).dt.days
)
# bucket into 0-1, 1-3, 3-7, 7-14, 14-30, 30-90, 90+ days before the alert
bins = [-1, 1, 3, 7, 14, 30, 90, np.inf]
labels = ["0-1d", "1-3d", "3-7d", "7-14d", "14-30d", "30-90d", "90d+"]
bucketed = pd.cut(days_before, bins=bins, labels=labels)
pre_signal_activity = (
    bucketed.value_counts().reindex(labels).rename_axis("days_before")
    .reset_index(name="count")
)
pre_signal_activity.to_csv(OUT_DIR / "pre_signal_activity.csv", index=False)

# ---------------------------------------------------------------- #
# 6. alert_behavior.csv  (REQUIRED: escalated vs dismissed comparison)
#    -- lightweight per-alert feature summary for TRAIN alerts only,
#    since this is the only set with a known target.
# ---------------------------------------------------------------- #
train_tx_f = train_tx.merge(train_signals[["signal_id", "signal_sanasi"]], on="signal_id", how="inner")
train_tx_f = train_tx_f[train_tx_f["tranzaksiya_vaqti"] < train_tx_f["signal_sanasi"]].copy()
train_tx_f["days_before"] = (train_tx_f["signal_sanasi"] - train_tx_f["tranzaksiya_vaqti"]).dt.total_seconds() / 86400
train_tx_f["is_in"] = (train_tx_f["kirim_chiqim"] == "kirim").astype(int)
train_tx_f["is_out"] = (train_tx_f["kirim_chiqim"] == "chiqim").astype(int)
train_tx_f["is_cash"] = (train_tx_f["tranzaksiya_turi"] == "naqd").astype(int)
train_tx_f["is_intl"] = (train_tx_f["tranzaksiya_turi"] == "xalqaro").astype(int)

agg = train_tx_f.groupby("signal_id").agg(
    transaction_count=("signal_id", "size"),
    amount_sum=("miqdor_indeksi", "sum"),
    amount_mean=("miqdor_indeksi", "mean"),
    amount_max=("miqdor_indeksi", "max"),
    incoming_count=("is_in", "sum"),
    outgoing_count=("is_out", "sum"),
    cash_count=("is_cash", "sum"),
    international_count=("is_intl", "sum"),
    days_since_last_transaction=("days_before", "min"),
)
denom = agg["transaction_count"].replace(0, np.nan)
agg["incoming_ratio"] = agg["incoming_count"] / denom
agg["outgoing_ratio"] = agg["outgoing_count"] / denom
agg["cash_ratio"] = agg["cash_count"] / denom
agg["international_ratio"] = agg["international_count"] / denom

# simple burstiness / trend per signal (kept lightweight for the EDA summary;
# the full model pipeline computes richer versions of these)
def _burst_trend(g):
    d = g.sort_values("tranzaksiya_vaqti")
    days_before = d["days_before"].to_numpy()
    amounts = d["miqdor_indeksi"].to_numpy()
    n = len(amounts)
    out = {}
    if n > 2:
        gaps = np.diff(np.sort(days_before)[::-1])
        gaps = -gaps[gaps <= 0] if len(gaps) else np.array([0.0])
        out["gap_cv"] = gaps.std() / gaps.mean() if gaps.mean() > 1e-9 else 0.0
    else:
        out["gap_cv"] = 0.0
    if n >= 3:
        k = min(10, n)
        idx = np.argsort(days_before)[:k]
        x = -days_before[idx]
        yv = amounts[idx]
        x = x - x.mean()
        out["recent_amount_trend"] = (x * (yv - yv.mean())).sum() / (x @ x) if (x @ x) > 1e-9 else 0.0
    else:
        out["recent_amount_trend"] = 0.0
    hist_mean, hist_std = amounts.mean(), amounts.std(ddof=0)
    last_amt = amounts[np.argmin(days_before)]
    out["last_tx_zscore"] = (last_amt - hist_mean) / hist_std if hist_std > 1e-9 else 0.0
    return pd.Series(out)

bt = train_tx_f.groupby("signal_id").apply(_burst_trend)
agg = agg.join(bt, how="left")

alert_behavior = agg.reset_index().merge(
    train_signals[["signal_id", "eskalatsiya"]], on="signal_id", how="left"
)
alert_behavior.to_csv(OUT_DIR / "alert_behavior.csv", index=False)

print("Wrote all eda_data/*.csv files:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(" -", f.name)

print(
    "\nNOTE: feature_importance.csv and model_comparison.csv are not "
    "generated here -- write those out directly from your training script "
    "(e.g. `imp.to_csv('eda_data/feature_importance.csv')` and the per-model "
    "OOF AUC scores) so the numbers shown on the site match your real run."
)

Wrote all eda_data/*.csv files:
 - activity_over_time.csv
 - alert_behavior.csv
 - feature_importance.csv
 - model_comparison.csv
 - pre_signal_activity.csv
 - target_distribution.csv
 - transaction_direction.csv
 - transaction_types.csv

NOTE: feature_importance.csv and model_comparison.csv are not generated here -- write those out directly from your training script (e.g. `imp.to_csv('eda_data/feature_importance.csv')` and the per-model OOF AUC scores) so the numbers shown on the site match your real run.


In [ ]:
"""
Escalation-probability model, v2.
Improvements over the baseline:
  1. Single combined groupby instead of ~6 redundant passes over `tx`.
  2. Trend / anomaly / burstiness features (not just level features):
     - recent-vs-historical amount ratio (is the account "hotter" than usual)
     - per-account z-score of the single largest recent transaction
     - inter-transaction gap statistics (burstiness)
     - linear trend slope of transaction size over the last N events
  3. Feature-importance based pruning to cut correlated/noisy window columns.
  4. Ensemble: CatBoost + LightGBM + Logistic Regression, each seed-bagged,
     combined via rank-averaging (more robust than probability-averaging for AUC).
  5. Two CV views printed: stratified-random (matches this competition's actual
     train/test split, based on the signal_id samples reviewed) and a
     chronological check, purely as a sanity diagnostic.

Usage: adjust the DATA_DIR path, then run end-to-end.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata

from catboost import CatBoostClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("lightgbm not installed -> skipping LGBM leg of the ensemble "
          "(pip install lightgbm to enable it).")

DATA_DIR = "data"
N_FOLDS = 5
SEEDS = [42, 202, 777]          # seed-bagging: average across these for each model
IMPORTANCE_KEEP_FRAC = 0.85     # keep top X% cumulative importance mass


# --------------------------------------------------------------------------- #
# Load
# --------------------------------------------------------------------------- #
train_signals = pd.read_csv(f"{DATA_DIR}/train_signals.csv")
test_signals = pd.read_csv(f"{DATA_DIR}/test_signals.csv")
train_transactions = pd.read_parquet(f"{DATA_DIR}/train_transactions.parquet")
test_transactions = pd.read_parquet(f"{DATA_DIR}/test_transactions.parquet")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

for df, col in [(train_signals, "signal_sanasi"), (test_signals, "signal_sanasi"),
                 (train_transactions, "tranzaksiya_vaqti"), (test_transactions, "tranzaksiya_vaqti")]:
    df[col] = pd.to_datetime(df[col])

print("TRAIN SIGNALS:", train_signals.shape, " TEST SIGNALS:", test_signals.shape)
print("TRAIN TX:", train_transactions.shape, " TEST TX:", test_transactions.shape)


# --------------------------------------------------------------------------- #
# Feature engineering
# --------------------------------------------------------------------------- #
def create_features(signals, transactions):
    tx = transactions.merge(signals[["signal_id", "signal_sanasi"]], on="signal_id", how="inner")
    tx = tx[tx["tranzaksiya_vaqti"] < tx["signal_sanasi"]].copy()
    tx["days_before"] = (tx["signal_sanasi"] - tx["tranzaksiya_vaqti"]).dt.total_seconds() / 86400

    tx["is_in"] = (tx["kirim_chiqim"] == "kirim").astype("int8")
    tx["is_out"] = (tx["kirim_chiqim"] == "chiqim").astype("int8")
    tx["is_card"] = (tx["tranzaksiya_turi"] == "karta").astype("int8")
    tx["is_bank"] = (tx["tranzaksiya_turi"] == "bank_otkazmasi").astype("int8")
    tx["is_cash"] = (tx["tranzaksiya_turi"] == "naqd").astype("int8")
    tx["is_intl"] = (tx["tranzaksiya_turi"] == "xalqaro").astype("int8")

    tx["in_amt"] = np.where(tx["is_in"] == 1, tx["miqdor_indeksi"], 0.0)
    tx["out_amt"] = np.where(tx["is_out"] == 1, tx["miqdor_indeksi"], 0.0)
    tx["intl_amt"] = np.where(tx["is_intl"] == 1, tx["miqdor_indeksi"], 0.0)
    tx["cash_amt"] = np.where(tx["is_cash"] == 1, tx["miqdor_indeksi"], 0.0)

    # -------- one combined groupby for all base level/direction/type stats -----
    features = tx.groupby("signal_id").agg(
        transaction_count=("signal_id", "size"),
        amount_sum=("miqdor_indeksi", "sum"),
        amount_mean=("miqdor_indeksi", "mean"),
        amount_std=("miqdor_indeksi", "std"),
        amount_min=("miqdor_indeksi", "min"),
        amount_max=("miqdor_indeksi", "max"),
        first_transaction=("tranzaksiya_vaqti", "min"),
        last_transaction=("tranzaksiya_vaqti", "max"),
        active_days=("tranzaksiya_vaqti", lambda x: x.dt.date.nunique()),
        incoming_count=("is_in", "sum"),
        outgoing_count=("is_out", "sum"),
        card_count=("is_card", "sum"),
        bank_transfer_count=("is_bank", "sum"),
        cash_count=("is_cash", "sum"),
        international_count=("is_intl", "sum"),
        incoming_amount=("in_amt", "sum"),
        outgoing_amount=("out_amt", "sum"),
        international_amount=("intl_amt", "sum"),
        cash_amount=("cash_amt", "sum"),
        days_since_last_transaction=("days_before", "min"),
        days_since_first_transaction=("days_before", "max"),
    )

    quantiles = (
        tx.groupby("signal_id")["miqdor_indeksi"]
        .quantile([0.25, 0.50, 0.75, 0.90])
        .unstack()
        .rename(columns={0.25: "amount_q25", 0.50: "amount_median",
                          0.75: "amount_q75", 0.90: "amount_q90"})
    )
    features = features.join(quantiles, how="left")

    for days in [7, 30, 90]:
        recent = tx[tx["days_before"] <= days]
        if len(recent) == 0:
            continue
        rf = recent.groupby("signal_id").agg(**{
            f"tx_last_{days}d": ("signal_id", "size"),
            f"amount_last_{days}d": ("miqdor_indeksi", "sum"),
            f"amount_mean_last_{days}d": ("miqdor_indeksi", "mean"),
            f"out_last_{days}d": ("is_out", "sum"),
            f"intl_last_{days}d": ("is_intl", "sum"),
        })
        features = features.join(rf, how="left")

    
    tx_sorted = tx.sort_values(["signal_id", "tranzaksiya_vaqti"])

    def _trailing_stats(group):
        amounts = group["miqdor_indeksi"].to_numpy()
        days_before = group["days_before"].to_numpy()
        n = len(amounts)

        out = {}
        # anomaly: how big is the single most-recent transaction vs this
        # account's own historical mean/std (per-account z-score)
        hist_mean = amounts.mean()
        hist_std = amounts.std(ddof=0) if n > 1 else 0.0
        last_amt = amounts[np.argmin(days_before)]  # most recent = smallest days_before
        out["last_tx_zscore"] = (last_amt - hist_mean) / hist_std if hist_std > 1e-9 else 0.0
        out["max_to_mean_ratio"] = amounts.max() / hist_mean if hist_mean > 1e-9 else 0.0

        # burstiness: variability of gaps between consecutive transactions
        if n > 2:
            gaps = -np.diff(np.sort(days_before)[::-1])  # chronological gaps in days
            gaps = gaps[gaps >= 0]
            if len(gaps) > 1 and gaps.mean() > 1e-9:
                out["gap_cv"] = gaps.std() / gaps.mean()  # coefficient of variation
            else:
                out["gap_cv"] = 0.0
        else:
            out["gap_cv"] = 0.0

        # trend: slope of amount vs. recency over the last up-to-10 events
        k = min(10, n)
        if k >= 3:
            recent_idx = np.argsort(days_before)[:k]          # k most recent
            x = -days_before[recent_idx]                      # increasing "time"
            yv = amounts[recent_idx]
            x = x - x.mean()
            slope = (x * (yv - yv.mean())).sum() / (x @ x) if (x @ x) > 1e-9 else 0.0
            out["recent_amount_trend"] = slope
        else:
            out["recent_amount_trend"] = 0.0

        # recent-vs-historical activity ratio: last-7d volume vs. average
        # weekly volume across the whole observed history
        span_days = max(days_before.max(), 1.0)
        weekly_avg_count = n / (span_days / 7.0)
        recent7_count = (days_before <= 7).sum()
        out["activity_vs_own_baseline"] = recent7_count / weekly_avg_count if weekly_avg_count > 1e-9 else 0.0

        return pd.Series(out)

    trend_features = tx_sorted.groupby("signal_id").apply(_trailing_stats)
    features = features.join(trend_features, how="left")

    # -------------------- span / calendar / ratios -----------------------------
    features["history_span_days"] = (features["last_transaction"] - features["first_transaction"]).dt.total_seconds() / 86400

    signal_dates = signals[["signal_id", "signal_sanasi"]].set_index("signal_id")
    features = features.join(signal_dates, how="right")
    features["signal_month"] = features["signal_sanasi"].dt.month
    features["signal_dayofweek"] = features["signal_sanasi"].dt.dayofweek
    # NOTE: signal_year / dayofyear deliberately dropped -- both years appear
    # in train and test, but raw year/day-of-year rarely generalizes and
    # mostly adds overfitting surface for a feature that isn't causal.

    denom = features["transaction_count"].replace(0, np.nan)
    for num, name in [("incoming_count", "incoming_ratio"), ("outgoing_count", "outgoing_ratio"),
                        ("international_count", "international_ratio"), ("cash_count", "cash_ratio"),
                        ("card_count", "card_ratio"), ("bank_transfer_count", "bank_transfer_ratio")]:
        features[name] = features[num] / denom

    total_amt = features["amount_sum"].replace(0, np.nan)
    for num, name in [("incoming_amount", "incoming_amount_ratio"), ("outgoing_amount", "outgoing_amount_ratio"),
                        ("international_amount", "international_amount_ratio")]:
        features[name] = features[num] / total_amt

    features["transactions_per_active_day"] = features["transaction_count"] / features["active_days"].replace(0, np.nan)

    if "tx_last_7d" in features.columns and "tx_last_30d" in features.columns:
        features["activity_7d_vs_30d"] = features["tx_last_7d"] / features["tx_last_30d"].replace(0, np.nan)

    features = features.drop(columns=["signal_sanasi", "first_transaction", "last_transaction"], errors="ignore")
    features = features.reset_index().replace([np.inf, -np.inf], np.nan)
    return features


print("\nBuilding train features...")
train_features = create_features(train_signals, train_transactions)
print("Building test features...")
test_features = create_features(test_signals, test_transactions)
print("Train feature shape:", train_features.shape, " Test feature shape:", test_features.shape)

train_features = train_features.merge(train_signals[["signal_id", "eskalatsiya"]], on="signal_id", how="left")

X = train_features.drop(columns=["signal_id", "eskalatsiya"])
y = train_features["eskalatsiya"]
X_test = test_features.drop(columns=["signal_id"]).reindex(columns=X.columns)

print("\nInitial feature count:", X.shape[1])
print("Target distribution:\n", y.value_counts(normalize=True))


# --------------------------------------------------------------------------- #
# Stage 1: quick CatBoost pass purely to rank feature importance, then prune
# --------------------------------------------------------------------------- #
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

probe = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6, eval_metric="AUC",
    loss_function="Logloss", random_seed=42, verbose=False, allow_writing_files=False,
)
probe.fit(X, y)
imp = pd.Series(probe.get_feature_importance(), index=X.columns).sort_values(ascending=False)
cum = imp.cumsum() / imp.sum()
keep_cols = cum[cum <= IMPORTANCE_KEEP_FRAC].index.tolist()
if len(keep_cols) < 15:  # safety floor
    keep_cols = imp.head(15).index.tolist()

print(f"\nPruned feature set: {len(keep_cols)} / {X.shape[1]} kept "
      f"(cumulative importance <= {IMPORTANCE_KEEP_FRAC})")
print("Top 15 features:\n", imp.head(15))

X = X[keep_cols]
X_test = X_test[keep_cols]


# --------------------------------------------------------------------------- #
# Stage 2: full CV with seed-bagged CatBoost + LightGBM + Logistic Regression
# --------------------------------------------------------------------------- #
oof_cb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_lr = np.zeros(len(X))
test_cb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_lr = np.zeros(len(X_test))

X_filled = X.fillna(X.median())
X_test_filled = X_test.fillna(X.median())  # impute test with TRAIN medians, not test's own

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    print(f"\n================ FOLD {fold} ================")
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    Xf_tr, Xf_va = X_filled.iloc[tr_idx], X_filled.iloc[va_idx]

    # ---- CatBoost, seed-bagged ----
    cb_va_pred = np.zeros(len(va_idx))
    cb_te_pred = np.zeros(len(X_test))
    for seed in SEEDS:
        model = CatBoostClassifier(
            iterations=2500, learning_rate=0.03, depth=7, l2_leaf_reg=5,
            eval_metric="AUC", loss_function="Logloss", early_stopping_rounds=150,
            random_seed=seed + fold, verbose=False, allow_writing_files=False,
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
        cb_va_pred += model.predict_proba(X_va)[:, 1] / len(SEEDS)
        cb_te_pred += model.predict_proba(X_test)[:, 1] / (len(SEEDS) * N_FOLDS)
    oof_cb[va_idx] = cb_va_pred
    test_cb += cb_te_pred
    print(f"  CatBoost fold AUC:  {roc_auc_score(y_va, cb_va_pred):.6f}")

    # ---- LightGBM, seed-bagged ----
    if HAS_LGBM:
        lgb_va_pred = np.zeros(len(va_idx))
        lgb_te_pred = np.zeros(len(X_test))
        for seed in SEEDS:
            lgb_model = LGBMClassifier(
                n_estimators=2000, learning_rate=0.03, max_depth=7, num_leaves=63,
                reg_lambda=5.0, subsample=0.8, colsample_bytree=0.8,
                random_state=seed + fold, verbose=-1,
            )
            lgb_model.fit(
                X_tr, y_tr, eval_set=[(X_va, y_va)],
                callbacks=[],
            )
            lgb_va_pred += lgb_model.predict_proba(X_va)[:, 1] / len(SEEDS)
            lgb_te_pred += lgb_model.predict_proba(X_test)[:, 1] / (len(SEEDS) * N_FOLDS)
        oof_lgb[va_idx] = lgb_va_pred
        test_lgb += lgb_te_pred
        print(f"  LightGBM fold AUC:  {roc_auc_score(y_va, lgb_va_pred):.6f}")

    # ---- Logistic Regression (scaled, median-imputed) for ensemble diversity ----
    scaler = StandardScaler()
    Xf_tr_s = scaler.fit_transform(Xf_tr)
    Xf_va_s = scaler.transform(Xf_va)
    Xf_test_s = scaler.transform(X_test_filled)
    lr = LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced")
    lr.fit(Xf_tr_s, y_tr)
    lr_va_pred = lr.predict_proba(Xf_va_s)[:, 1]
    oof_lr[va_idx] = lr_va_pred
    test_lr += lr.predict_proba(Xf_test_s)[:, 1] / N_FOLDS
    print(f"  LogReg fold AUC:    {roc_auc_score(y_va, lr_va_pred):.6f}")


# --------------------------------------------------------------------------- #
# Rank-average ensemble (robust to differing probability scales across models)
# --------------------------------------------------------------------------- #
def to_rank(a):
    return rankdata(a) / len(a)

if HAS_LGBM:
    oof_blend = 0.45 * to_rank(oof_cb) + 0.40 * to_rank(oof_lgb) + 0.15 * to_rank(oof_lr)
    test_blend = 0.45 * to_rank(test_cb) + 0.40 * to_rank(test_lgb) + 0.15 * to_rank(test_lr)
else:
    oof_blend = 0.75 * to_rank(oof_cb) + 0.25 * to_rank(oof_lr)
    test_blend = 0.75 * to_rank(test_cb) + 0.25 * to_rank(test_lr)

print("\n================ OVERALL RESULTS ================")
print("CatBoost   OOF AUC:", roc_auc_score(y, oof_cb))
if HAS_LGBM:
    print("LightGBM   OOF AUC:", roc_auc_score(y, oof_lgb))
print("LogReg     OOF AUC:", roc_auc_score(y, oof_lr))
print("BLEND      OOF AUC:", roc_auc_score(y, oof_blend))
print("==================================================")


# --------------------------------------------------------------------------- #
# Submission
# --------------------------------------------------------------------------- #
predictions = pd.DataFrame({
    "signal_id": test_signals["signal_id"],
    "ehtimollik": test_blend,
})
predictions["ehtimollik"] = predictions["ehtimollik"].clip(0, 1)

assert set(predictions["signal_id"]) == set(test_signals["signal_id"])
assert predictions["ehtimollik"].between(0, 1).all()
assert predictions.columns.tolist() == ["signal_id", "ehtimollik"]
assert predictions["signal_id"].duplicated().sum() == 0

TEAM_ID = "B30D79CF"
filename = f"team_{TEAM_ID}.csv"
predictions.to_csv(filename, index=False, float_format="%.6f")
print(f"\nFINAL FILE CREATED: {filename}")
print(predictions.describe())

lightgbm not installed -> skipping LGBM leg of the ensemble (pip install lightgbm to enable it).
TRAIN SIGNALS: (14000, 3)  TEST SIGNALS: (6000, 2)
TRAIN TX: (6987663, 5)  TEST TX: (3027575, 5)

Building train features...
Building test features...
Train feature shape: (14000, 58)  Test feature shape: (6000, 58)

Initial feature count: 57
Target distribution:
 eskalatsiya
0    0.828214
1    0.171786
Name: proportion, dtype: float64

Pruned feature set: 39 / 57 kept (cumulative importance <= 0.85)
Top 15 features:
 cash_amount                 6.082804
amount_min                  6.015612
last_tx_zscore              4.135749
amount_max                  3.610297
recent_amount_trend         3.123174
cash_ratio                  2.885397
activity_vs_own_baseline    2.870165
amount_q25                  2.675087
gap_cv                      2.569168
amount_std                  2.222997
activity_7d_vs_30d          2.204904
bank_transfer_ratio         2.202131
international_ratio         2.101964
